# Synthetic Parameter Recovery

Validates the full inference pipeline by generating synthetic data from known GP parameters
and checking whether VBMC recovers those parameters.

## Pipeline
1. Fix ground-truth `(length_scale, mu_0)`
2. Sample pseudocoherences `y_train`, `y_test` from the GP prior
3. Sample utterances `u_train` from the RSA speaker given `y_train`
4. Sample synthetic prevalence ratings from the Beta mixture given `y_test`
5. Run VBMC to recover `(length_scale, mu_0)`
6. Check posterior covers ground truth

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import sys, pickle as pkl
sys.path.insert(0, ".")

import numpy as np
import jax
import jax.numpy as jnp
import blackjax
import matplotlib.pyplot as plt
from scipy.special import logsumexp as scipy_logsumexp
from pyvbmc import VBMC

from model_jax import (
    rbf_kernel,
    p_u_given_y,
    make_log_density_fn_joint,
    fit_beta_mixtures_all_features,
    beta_mixture_log_likelihood,
)

print(f"backend: {jax.default_backend()}")

## Ground-truth parameters and feature embeddings

In [ ]:
# Ground-truth parameters to recover
TRUE_PARAMS = {
    'length_scale': 0.4,
    'mu_0':         0.0,
    'output_scale': 1.5,
    'beta':         3.0,
}

# Use real feature embeddings from the dataframe so geometry is realistic
with open('../features/set2_features_dataframe.pkl', 'rb') as f:
    df = pkl.load(f)

train_df = df[df.split == 'train']
feat_idx = df.set_index('feature')

TEST_FEATURE_NAMES = [
    'can eat spicy food', 'eat breakfast very late', 'eat five meals a day',
    'like juice with pulp', 'put pepper on all their foods',
    'cry easily', 'like to collect rocks', 'like to dance',
    'like to give high-fives', 'like to read books',
    'can roll their tongue', 'can snap with their toes',
    'can wiggle their ears', 'have cold hands and feet', 'snore when they sleep',
]

x_train = jnp.array(train_df[['x_2d', 'y_2d']].values)   # (45, 2)
x_test  = jnp.array(
    [feat_idx.loc[name, ['x_2d', 'y_2d']].values for name in TEST_FEATURE_NAMES]
)  # (15, 2)

n_train = x_train.shape[0]
J       = x_test.shape[0]
print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")

## Step 1: Sample pseudocoherences from the GP prior

In [ ]:
rng = np.random.default_rng(42)

# Stack all features: test first then train (matches make_log_density_fn_joint convention)
X_all = jnp.vstack([x_test, x_train])   # (J + n_train, 2)
mu    = jnp.full(J + n_train, TRUE_PARAMS['mu_0'])
K     = rbf_kernel(X_all, TRUE_PARAMS['length_scale'], TRUE_PARAMS['output_scale'])

y_all = rng.multivariate_normal(np.array(mu), np.array(K))  # (J + n_train,)
y_test_true  = y_all[:J]           # (J,)       pseudocoherences for test features
y_train_true = y_all[J:]           # (n_train,) pseudocoherences for training features

pz1_test_true = 1 / (1 + np.exp(-y_test_true))   # (J,)

print(f"y_train range: [{y_train_true.min():.2f}, {y_train_true.max():.2f}]")
print(f"y_test  range: [{y_test_true.min():.2f}, {y_test_true.max():.2f}]")
print(f"pz1_test range: [{pz1_test_true.min():.2f}, {pz1_test_true.max():.2f}]")

## Step 2: Sample utterances from RSA speaker

In [ ]:
# For each training feature, sample P(u|y) from the speaker and draw one utterance
beta = TRUE_PARAMS['beta']
u_probs = np.array([
    float(p_u_given_y(0, float(y_i), beta))   # P(generic | y_i)
    for y_i in y_train_true
])  # (n_train,) — probability of generic utterance

# Sample: 0=generic, 1=specific
u_train_true = (rng.uniform(size=n_train) > u_probs).astype(np.int32)
u_train      = jnp.array(u_train_true)

print(f"Utterances: {(u_train_true==0).sum()} generic, {(u_train_true==1).sum()} specific")
print(f"P(generic) range: [{u_probs.min():.2f}, {u_probs.max():.2f}]")

## Step 3: Sample synthetic prevalence ratings from the Beta mixture

In [ ]:
# Ground-truth Beta mixture params — use the same shape as the real data fits
# kind-linked component: Beta(5, 1) — high prevalence
# non-kind-linked component: Beta(1, 5) — low prevalence
ALPHA_KL, BETA_KL   = 5.0, 1.0
ALPHA_NKL, BETA_NKL = 1.0, 5.0

N_PARTICIPANTS = 100  # synthetic participants per test feature

ratings = np.zeros((N_PARTICIPANTS, J))
for j in range(J):
    pz1 = pz1_test_true[j]
    for n in range(N_PARTICIPANTS):
        # draw component assignment
        if rng.uniform() < pz1:
            ratings[n, j] = rng.beta(ALPHA_KL, BETA_KL)
        else:
            ratings[n, j] = rng.beta(ALPHA_NKL, BETA_NKL)

responses = jnp.array(ratings)  # (N, J)
N = N_PARTICIPANTS
print(f"responses: {responses.shape}")
print(f"empirical mean range: [{float(responses.mean(0).min()):.2f}, {float(responses.mean(0).max()):.2f}]")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(J), np.array(responses.mean(0)), color='steelblue', alpha=0.7, label='synthetic mean')
ax.plot(range(J), pz1_test_true, 'ro', label='true pz1')
ax.set_xticks(range(J))
ax.set_xticklabels(TEST_FEATURE_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('prevalence')
ax.set_title('Synthetic ratings: empirical mean vs true pz1')
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Run VBMC to recover `(length_scale, mu_0)`

In [ ]:
FIXED_PARAMS = {
    'output_scale': TRUE_PARAMS['output_scale'],
    'beta':         TRUE_PARAMS['beta'],
}

N_WARMUP  = 300
N_SAMPLES = 1000
STEP      = 5

In [ ]:
eval_count = [0]
eval_log   = []

def log_joint(phi):
    log_ls, mu_0 = float(phi[0]), float(phi[1])
    length_scale = float(np.exp(log_ls))

    eval_count[0] += 1
    print(f"  eval {eval_count[0]:3d}: ls={length_scale:.3f}  mu_0={mu_0:.3f}", end="  ")

    log_prior_ls  = float(-0.5 * ((log_ls - np.log(0.5)) / 1.5) ** 2)
    log_prior_mu0 = float(-0.5 * mu_0 ** 2)
    log_prior = log_prior_ls + log_prior_mu0

    params = {**FIXED_PARAMS, 'length_scale': length_scale, 'mu_0': mu_0}
    log_density_fn = make_log_density_fn_joint(u_train, x_train, x_test, params)

    init_position = {
        'training_coherences': jnp.zeros(n_train),
        'test_coherences':     jnp.zeros(J),
    }

    rng_key = jax.random.PRNGKey(0)
    rng_key, warmup_key = jax.random.split(rng_key)
    warmup = blackjax.window_adaptation(blackjax.nuts, log_density_fn)
    (state, nuts_params), _ = warmup.run(warmup_key, init_position, num_steps=N_WARMUP)

    nuts = blackjax.nuts(log_density_fn, **nuts_params)

    @jax.jit
    def one_step(state, key):
        return nuts.step(key, state)

    keys = jax.random.split(rng_key, N_SAMPLES)
    test_coherences_all = []
    for key in keys[:-1]:
        state, _ = one_step(state, key)
        test_coherences_all.append(np.array(state.position['test_coherences']))
    test_coherences_all = np.array(test_coherences_all)

    log_liks_s = []
    for s in range(0, test_coherences_all.shape[0], STEP):
        pz1_s    = jnp.array(jax.nn.sigmoid(test_coherences_all[s]))
        pz1_s_bc = jnp.tile(pz1_s, (N, 1))
        beta_params_s = fit_beta_mixtures_all_features(responses, pz1_s_bc)
        log_liks_s.append(float(beta_mixture_log_likelihood(responses, pz1_s, beta_params_s)))

    log_lik   = float(np.median(log_liks_s))
    noise_std = float(np.std(log_liks_s))

    if not np.isfinite(log_lik):
        print(f"  [WARNING: non-finite log_lik, clamping]")
        log_lik, noise_std = -1e6, 1.0

    log_joint_val = float(log_lik + log_prior)
    print(f"log_lik={log_lik:.1f}  noise_std={noise_std:.2f}  log_joint={log_joint_val:.1f}")
    eval_log.append((np.array(phi, dtype=float), log_joint_val, noise_std))
    return log_joint_val, noise_std

In [ ]:
x0  = np.array([np.log(0.4), 0.0])   # start near true params
lb  = np.array([np.log(0.05), -4.0])
ub  = np.array([np.log(20.0),  4.0])
plb = np.array([np.log(0.1), -2.0])
pub = np.array([np.log(5.0),  2.0])

print(f"True params: ls={TRUE_PARAMS['length_scale']}, mu_0={TRUE_PARAMS['mu_0']}")
print(f"Starting at: ls={np.exp(x0[0]):.2f}, mu_0={x0[1]:.2f}")
print()

vbmc = VBMC(log_joint, x0, lb, ub, plb, pub,
            options={'specify_target_noise': True})
vbmc_result, vbmc_stats = vbmc.optimize()

## Step 5: Check parameter recovery

In [ ]:
phi_samples, _ = vbmc_result.vp.sample(int(1e4))
ls_samples  = np.exp(phi_samples[:, 0])
mu0_samples = phi_samples[:, 1]

print(f"True:          ls={TRUE_PARAMS['length_scale']:.3f}  mu_0={TRUE_PARAMS['mu_0']:.3f}")
print(f"Posterior mean: ls={ls_samples.mean():.3f}  mu_0={mu0_samples.mean():.3f}")
print(f"Posterior std:  ls={ls_samples.std():.3f}  mu_0={mu0_samples.std():.3f}")
print(f"95% CI ls:   [{np.percentile(ls_samples, 2.5):.3f}, {np.percentile(ls_samples, 97.5):.3f}]  "
      f"covers true? {np.percentile(ls_samples, 2.5) < TRUE_PARAMS['length_scale'] < np.percentile(ls_samples, 97.5)}")
print(f"95% CI mu_0: [{np.percentile(mu0_samples, 2.5):.3f}, {np.percentile(mu0_samples, 97.5):.3f}]  "
      f"covers true? {np.percentile(mu0_samples, 2.5) < TRUE_PARAMS['mu_0'] < np.percentile(mu0_samples, 97.5)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(ls_samples, bins=50, density=True, color='steelblue', alpha=0.7)
axes[0].axvline(TRUE_PARAMS['length_scale'], color='tomato', lw=2, linestyle='--', label=f"true={TRUE_PARAMS['length_scale']}")
axes[0].axvline(np.median(ls_samples), color='navy', lw=1.5, label=f'posterior median={np.median(ls_samples):.2f}')
axes[0].set_xlabel('length_scale')
axes[0].set_title('Recovery: length_scale')
axes[0].legend(fontsize=8)

axes[1].hist(mu0_samples, bins=50, density=True, color='seagreen', alpha=0.7)
axes[1].axvline(TRUE_PARAMS['mu_0'], color='tomato', lw=2, linestyle='--', label=f"true={TRUE_PARAMS['mu_0']}")
axes[1].axvline(np.median(mu0_samples), color='darkgreen', lw=1.5, label=f'posterior median={np.median(mu0_samples):.2f}')
axes[1].set_xlabel('mu_0')
axes[1].set_title('Recovery: mu_0')
axes[1].legend(fontsize=8)

axes[2].scatter(ls_samples[::10], mu0_samples[::10], alpha=0.1, s=2, color='purple')
axes[2].axvline(TRUE_PARAMS['length_scale'], color='tomato', lw=1.5, linestyle='--')
axes[2].axhline(TRUE_PARAMS['mu_0'], color='tomato', lw=1.5, linestyle='--')
axes[2].set_xlabel('length_scale')
axes[2].set_ylabel('mu_0')
axes[2].set_title('Joint posterior (red lines = true params)')

plt.suptitle(f"Parameter recovery: true ls={TRUE_PARAMS['length_scale']}, mu_0={TRUE_PARAMS['mu_0']}", y=1.02)
plt.tight_layout()
plt.show()